# 🎬 KickTV — Canal de Televisión 24/7 en Google Colab

Este notebook te permite correr **KickTV** directamente desde Google Colab.

### ✅ Ventajas de correrlo en Colab
- **12 GB de RAM** gratis (el proyecto usa ~10 GB máximo)
- **CPU decente** para encoding con FFmpeg
- **Internet rápido** de Google para descargar videos y hacer streaming
- **Gratis** — sin necesidad de servidor propio

### ⚠️ Limitaciones importantes
- Las sesiones de Colab gratuito **se desconectan después de ~12 horas** (o antes si detectan inactividad)
- No hay GPU dedicada para encoding (se usa CPU, que es suficiente con `ultrafast`/`veryfast`)
- El dashboard web requiere un túnel (ngrok) para acceder desde fuera
- Los archivos se borran al terminar la sesión (pero se pueden montar en Google Drive)

---

## ▶️ Ejecuta las celdas en orden

## 1️⃣ Clonar el proyecto

Sube tu proyecto a GitHub (privado o público) y clónalo aquí.
Si no quieres usar GitHub, puedes subir un ZIP en la celda alternativa.

In [ ]:
# ============================================
# OPCIÓN A: Clonar desde GitHub
# ============================================
# Cambia la URL por la de tu repositorio
# Si es privado, usa: https://<TOKEN>@github.com/tu-usuario/kick.git

# !git clone https://github.com/tu-usuario/kick.git /content/kick

# ============================================
# OPCIÓN B: Subir un ZIP manualmente
# ============================================
# Descomenta estas líneas si prefieres subir un ZIP

# from google.colab import files
# uploaded = files.upload()  # Selecciona tu kick.zip
# !unzip -o kick.zip -d /content/kick

# ============================================
# OPCIÓN C: Desde Google Drive
# ============================================
# Monta Google Drive y copia el proyecto

from google.colab import drive
drive.mount('/content/drive')

# Copia el proyecto desde Drive (ajusta la ruta)
!cp -r "/content/drive/MyDrive/kick" /content/kick 2>/dev/null || echo "⚠️ No se encontró en Drive. Usa git clone o sube un ZIP."

# Verificar
!ls /content/kick/

## 2️⃣ Instalar dependencias del sistema

In [ ]:
# FFmpeg ya viene instalado en Colab, pero verificamos
!ffmpeg -version | head -1

# Verificar RAM disponible
import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"\n✅ RAM disponible: {ram_gb:.1f} GB")
if ram_gb >= 10:
    print("✅ RAM suficiente para KickTV")
else:
    print("⚠️ RAM baja, puede haber problemas")

## 3️⃣ Instalar dependencias de Python

In [ ]:
%cd /content/kick
!pip install -q -r requirements.txt
print("\n✅ Dependencias instaladas")

## 4️⃣ Configurar variables de entorno

⚠️ **IMPORTANTE**: Ingresa tu **Stream Key** de Kick y opcionalmente tus API keys.

In [ ]:
# ============================================
# ⚠️ CONFIGURA TUS DATOS AQUÍ
# ============================================

STREAM_URL = "rtmps://fa723fc1b171.global-contribute.live-video.net/app"  # No cambiar a menos que Kick cambie su servidor
STREAM_KEY = "tu_stream_key_aqui"  # 🔴 PON TU STREAM KEY AQUÍ

# API Keys (opcional, para más variedad de videos)
PEXELS_API_KEY = ""  # https://www.pexels.com/api/
PIXABAY_API_KEY = ""  # https://pixabay.com/api/docs/

# Encoding (Colab usa CPU, así que ultrafast es recomendado)
BITRATE = "3500k"     # Un poco más bajo para CPU de Colab
FPS = "30"
RESOLUTION = "1920x1080"
PRESET = "ultrafast"  # ⚡ Usar ultrafast en Colab para menos uso de CPU

# ============================================
# Generar archivo .env
# ============================================
env_content = f"""# KickTV — Google Colab Configuration
STREAM_URL={STREAM_URL}
STREAM_KEY={STREAM_KEY}

BITRATE={BITRATE}
FPS={FPS}
RESOLUTION={RESOLUTION}
PRESET={PRESET}
AUDIO_BITRATE=128k

DASHBOARD_HOST=0.0.0.0
DASHBOARD_PORT=8000

PEXELS_API_KEY={PEXELS_API_KEY}
PIXABAY_API_KEY={PIXABAY_API_KEY}

PROVIDER_LOCAL_ENABLED=true
PROVIDER_PEXELS_ENABLED={'true' if PEXELS_API_KEY else 'false'}
PROVIDER_PIXABAY_ENABLED={'true' if PIXABAY_API_KEY else 'false'}
PROVIDER_ARCHIVE_ENABLED=true
PROVIDER_YOUTUBE_ENABLED=false
PROVIDER_REDDIT_ENABLED=true

QUEUE_MIN_SIZE=5
QUEUE_MAX_HISTORY=500
VIDEO_CACHE_DIR=data/videos
MAX_VIDEO_DURATION=3600
MIN_VIDEO_DURATION=10

DATABASE_PATH=data/db/kicktv.db
LOG_DIR=logs
LOG_LEVEL=INFO
"""

with open("/content/kick/.env", "w") as f:
    f.write(env_content)

print("✅ Archivo .env creado")
if STREAM_KEY == "tu_stream_key_aqui":
    print("\n🔴 ¡ATENCIÓN! Necesitas poner tu STREAM KEY real arriba antes de continuar.")
else:
    print(f"✅ Stream Key configurada: {STREAM_KEY[:10]}...")

## 5️⃣ Crear directorios necesarios

In [ ]:
import os

dirs = [
    "data/videos",
    "data/db",
    "logs/app",
    "logs/ffmpeg",
    "logs/providers",
    "logs/errors",
]

for d in dirs:
    os.makedirs(f"/content/kick/{d}", exist_ok=True)

print("✅ Directorios creados")

## 6️⃣ (Opcional) Exponer el Dashboard con ngrok

El dashboard corre en el puerto 8000, pero Colab no permite acceso directo.
Usamos **ngrok** para crear un túnel público.

1. Crea una cuenta gratuita en [ngrok.com](https://ngrok.com)
2. Copia tu **authtoken** y pégalo abajo

In [ ]:
# ============================================
# Configurar ngrok (opcional pero recomendado)
# ============================================

NGROK_AUTH_TOKEN = ""  # 🔴 Pega tu token de ngrok aquí (opcional)

if NGROK_AUTH_TOKEN:
    !pip install -q pyngrok
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    tunnel = ngrok.connect(8000)
    print(f"\n🌐 Dashboard accesible en: {tunnel.public_url}")
    print(f"   (Ábrelo en tu navegador para ver el panel de control)")
else:
    print("⏭️ ngrok no configurado. El dashboard solo será accesible localmente.")
    print("   Para acceder al dashboard, configura NGROK_AUTH_TOKEN arriba.")
    print("\n   Alternativa: Usa la URL de Colab con el proxy:")
    print("   from google.colab.output import eval_js")
    print("   print(eval_js('google.colab.kernel.proxyPort(8000)'))")

## 7️⃣ 🚀 Iniciar KickTV

Esta celda inicia el servidor. **No la interrumpas** mientras quieras que el stream siga activo.

In [ ]:
%cd /content/kick

print("""
=========================================
|    KickTV - Google Colab Edition      |
|    24/7 Automatic Streaming           |
=========================================
""")

# Ejecutar el servidor
!python run.py

---

## 🛠️ Celdas de Utilidad

Ejecuta estas celdas según necesites (sin detener la celda principal).

In [ ]:
# ============================================
# Monitorear uso de recursos
# ============================================
import psutil

ram = psutil.virtual_memory()
cpu = psutil.cpu_percent(interval=1)
disk = psutil.disk_usage('/')

print(f"📊 Estado del Sistema:")
print(f"   CPU:  {cpu}%")
print(f"   RAM:  {ram.used / (1024**3):.1f} / {ram.total / (1024**3):.1f} GB ({ram.percent}%)")
print(f"   Disk: {disk.used / (1024**3):.1f} / {disk.total / (1024**3):.1f} GB ({disk.percent}%)")

In [ ]:
# ============================================
# Anti-desconexión (mantiene la sesión activa)
# ============================================
# Ejecuta esto en una celda separada para evitar
# que Colab desconecte la sesión por inactividad

import time
from IPython.display import display, Javascript

def keep_alive():
    """Simula actividad para evitar desconexión por inactividad."""
    while True:
        display(Javascript('console.log("keepalive " + new Date())'))
        time.sleep(60)  # Cada 60 segundos

# Descomentar para activar:
# keep_alive()

In [ ]:
# ============================================
# Ver logs recientes
# ============================================
!tail -50 /content/kick/logs/app/*.log 2>/dev/null || echo "No hay logs todavía"

In [ ]:
# ============================================
# Verificar estado del stream via API
# ============================================
import requests

try:
    r = requests.get("http://localhost:8000/api/status", timeout=5)
    data = r.json()
    print(f"📡 Estado: {data}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("   El servidor puede no estar corriendo aún.")

In [ ]:
# ============================================
# Guardar datos en Google Drive (backup)
# ============================================
# Útil para no perder la base de datos y configuración

!cp -r /content/kick/data /content/drive/MyDrive/kick_backup_data/ 2>/dev/null && echo "✅ Backup guardado en Drive" || echo "⚠️ Drive no montado"